# MMUbiPred training-label noise audit v1

This CPU-only notebook measures internal label contradictions and cross-source disagreement with the already-inspected official dbPTM archive. It fits no model, does not access the released independent test, and does not alter any dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Store a fine-grained read-only GitHub token in Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
print('No additional packages are required for this CPU-only audit.')

In [ ]:
import json

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
EXTERNAL_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v1')
EXTERNAL_BENCHMARK = EXTERNAL_ROOT / 'source/dbptm_ubiquitination_normalized.csv'
EXTERNAL_SEQUENCE_CACHE = EXTERNAL_ROOT / 'uniprot_sequences.json'
AUDIT_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/audits/training-label-noise-v1')
AUDIT_PATH = AUDIT_DIR / 'label_noise_audit.json'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
for required in [EXTERNAL_BENCHMARK, EXTERNAL_SEQUENCE_CACHE]:
    if not required.exists():
        raise FileNotFoundError(required)
print('All immutable source artifacts were found.')

## Run the deterministic audit

The output is persistent and an existing completed audit is not overwritten.

In [ ]:
if AUDIT_PATH.exists():
    print('Existing label-noise audit found; it was not repeated.')
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/audit_label_noise.py',
        '--data-dir', 'replication/MMUbiPred',
        '--external-benchmark', EXTERNAL_BENCHMARK,
        '--external-sequence-cache', EXTERNAL_SEQUENCE_CACHE,
        '--output', AUDIT_PATH,
    ])

In [ ]:
audit = json.loads(AUDIT_PATH.read_text())
internal = audit['released_training']
external = audit['external_cross_source_evidence']
display({
    'status': audit['status'],
    'scope': audit['scope'],
    'support': internal['support'],
    'exact_key_audits': internal['exact_key_audits'],
    'protein_label_composition': internal['protein_label_composition'],
    'negative_to_positive_proximity': internal['negative_to_positive_proximity'],
    'terminal_padding': internal['terminal_padding'],
})
display({
    'external_raw_support': external['raw_support'],
    'external_validated_support': external['validated_support'],
    'validation_reason_counts': external['validation_reason_counts'],
    'training_negative_vs_external_positive': external['training_negative_vs_external_positive'],
    'training_positive_vs_external_negative': external['training_positive_vs_external_negative'],
})

## Stop and review

Send both displayed dictionaries for interpretation before implementing a robust loss or positive–unlabeled objective.